In [1]:
import gc

In [2]:
import pandas as pd

label_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

def load_and_clean_chexpert(csv_path):
    # Load metadata
    df = pd.read_csv(csv_path)
    
    # Ensure numeric types and handle NaNs
    df[label_columns] = df[label_columns].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
    
    # Apply U-Zeros policy: Replace all -1 (uncertain) with 0 (negative)
    df[label_columns] = df[label_columns].replace(-1.0, 0.0)
    
    # Standardize image paths for the Kaggle environment
    df['Path'] = df['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
    df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")
    
    # Return cleaned dataframe containing only the Path and the 14 labels
    return pd.concat([df['Path'], df[label_columns]], axis=1)

# Generate cleanly separated dataframes
train_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/train.csv')
valid_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/valid.csv')

# Create the 5% subset for your convergence test
sampled_train_df = train_df.sample(frac=0.05, random_state=42).reset_index(drop=True)

print(f"Full Training Set: {len(train_df)}")
print(f"Sampled Training Set (5%): {len(sampled_train_df)}")
print(f"Validation Set: {len(valid_df)}")

Full Training Set: 223414
Sampled Training Set (5%): 11171
Validation Set: 234


In [3]:
import torch

# Calculate weights based on the FULL training set distribution
labels_df = train_df.iloc[:, 1:]

positive_counts = labels_df.sum(axis=0).values
total_samples = len(labels_df)
negative_counts = total_samples - positive_counts

epsilon = 1e-7
pos_weights_array = negative_counts / (positive_counts + epsilon)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weights = torch.tensor(pos_weights_array, dtype=torch.float32).to(device)

print("Calculated pos_weights for 14 classes:")
print(pos_weights)

Calculated pos_weights for 14 classes:
tensor([ 8.9823, 19.6903,  7.2746,  1.1160, 23.3211,  3.2762, 14.1129, 35.9952,
         5.6939, 10.4878,  1.5922, 62.4158, 23.7139,  0.9260], device='cuda:0')


In [4]:
# # Verify Fracture class presence in the validation set
# fracture_cases = int(dataset_valid['Fracture'].sum())
# total_cases = len(dataset_valid)
# fracture_ratio = (fracture_cases / total_cases) * 100

# print(f"Total validation images: {total_cases}")
# print(f"Positive Fracture cases: {fracture_cases}")
# print(f"Percentage of validation dataset: {fracture_ratio:.2f}%")

In [5]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import cv2

class CheXpertDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_path = row['Path']
        
        # Clean the string to extract just the relative path
        if raw_path.startswith('CheXpert-v1.0-small/'):
            relative_path = raw_path.replace('CheXpert-v1.0-small/', '')
        elif raw_path.startswith('/kaggle/input/chexpert/'):
            relative_path = raw_path.replace('/kaggle/input/chexpert/', '')
        else:
            relative_path = raw_path
            
        # Strip leading slashes so os.path.join operates correctly
        relative_path = relative_path.lstrip('/')
        
        image_path = os.path.join('/kaggle/input/datasets/ashery/chexpert/', relative_path)
        
        # # FIX: Use a context manager to prevent file handle memory leaks
        # with Image.open(image_path) as img:
        #     image = img.convert('RGB')

        img_array = cv2.imread(image_path)
        img_array = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(img_array) # Convert back to PIL only for torchvision transforms
        
        label = torch.tensor(row[1:].values.astype(float), dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [6]:
from torchvision import transforms

# Standard ImageNet normalization values required by EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CheXpertDataset(dataframe=train_df, transform=transform)
valid_dataset = CheXpertDataset(dataframe=valid_df, transform=transform)

# # Pass the sampled dataframe to the training dataset
# train_dataset = CheXpertDataset(dataframe=sampled_train_df, transform=transform)

# # Keep the full validation dataframe
# valid_dataset = CheXpertDataset(dataframe=valid_df, transform=transform)

In [7]:
from torch.utils.data import DataLoader

# train_dataset = CheXpertDataset(dataframe=dataset, transform=transform)
# valid_dataset = CheXpertDataset(dataframe=dataset_valid, transform=transform)

# Drastically increase batch size to saturate VRAM and reduce transfer overhead
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, 
    batch_size=256,              # Safe for GPU VRAM
    shuffle=True, 
    num_workers=0,               # NUCLEAR FIX: Disables multiprocessing
    pin_memory=False,            # NUCLEAR FIX: Disables shared memory (/dev/shm)
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=256,       
    shuffle=False, 
    num_workers=0,               # NUCLEAR FIX: Disables multiprocessing
    pin_memory=False             # NUCLEAR FIX: Disables shared memory (/dev/shm)
)
# Example: Iterate over the DataLoader
for images, labels in train_loader:
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    break

for images, labels in valid_loader:
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    break

Image batch shape: torch.Size([256, 3, 224, 224])
Label batch shape: torch.Size([256, 14])
Image batch shape: torch.Size([234, 3, 224, 224])
Label batch shape: torch.Size([234, 14])


In [8]:
import torch
import torch.nn as nn
import torchvision

class FocalLossMultiLabel(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        # Standard defaults from the original RetinaNet paper
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # torchvision's implementation takes raw logits (inputs) and float targets
        # It handles the sigmoid activation internally for numerical stability
        return torchvision.ops.sigmoid_focal_loss(
            inputs=inputs,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction=self.reduction
        )

In [9]:
import timm
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Load pretrained EfficientNet-B0 from timm (Pre-trained on ImageNet by default)
# num_classes=14 automatically replaces the 1000-class ImageNet head with a 14-class head
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=14)

# unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Unfreeze only the classifier head
for param in model.get_classifier().parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# if torch.cuda.device_count() > 1:
#     print(f"Utilizing {torch.cuda.device_count()} GPUs for training")
#     model = nn.DataParallel(model)

# Define optimizer and scheduler 
optimizer = AdamW(model.parameters(), lr=1e-5) # because we do all the network we kick the lr down a lil 
scheduler = CosineAnnealingLR(optimizer, T_max=10)

# Loss function (reusing your previously calculated pos_weights)
# criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
# criterion = nn.BCEWithLogitsLoss()

criterion = FocalLossMultiLabel(alpha=0.25, gamma=2.0, reduction='mean')

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [10]:
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings

# Suppress the scikit-learn UndefinedMetricWarning to keep your terminal clean
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.metrics')

def train_model(model, train_loader, valid_loader, optimizer, scheduler, criterion, device, epochs=5):
    scaler = torch.amp.GradScaler()

    # Class names for CheXpert Competition subset
    class_names = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        # =======================
        # TRAINING PHASE
        # =======================
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
        # for i, (images, labels) in enumerate(train_loader):
        #     if i >=5: break   
        
        # for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast(device_type="cuda"):
                # EfficientNet output is directly the logits (no .logits attribute like ViT)
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            train_loss += loss.item()

            del images, labels, outputs, loss

        train_loss /= len(train_loader)
        print(f"Training Loss: {train_loss:.4f}")

        # NEW: Force Python to dump all unreferenced RAM before validation starts
        gc.collect()
        torch.cuda.empty_cache() # Clears VRAM fragmentation

        # =======================
        # VALIDATION PHASE
        # =======================
        model.eval()
        valid_loss = 0.0
        
        # Lists to store all labels and predictions for the epoch
        all_labels = []
        all_preds = []

        with torch.no_grad():

            for images, labels in valid_loader:
            # for i, (images, labels) in enumerate(valid_loader):
            #     if i>=5 : break
                
            # for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)

                with torch.amp.autocast(device_type="cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                valid_loss += loss.item()

                # Apply sigmoid to convert logits to probabilities [0, 1]
                probs = torch.sigmoid(outputs)
                
                # Move to CPU and numpy for scikit-learn
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs.cpu().numpy())
                
                del images, labels, outputs, loss, probs

        valid_loss /= len(valid_loader)
        
        # Concatenate all batches
        all_labels = np.vstack(all_labels)
        all_preds = np.vstack(all_preds)

        # =======================
        # ROBUST METRICS CALCULATION
        # =======================
        per_class_roc_auc = []
        per_class_ap = []
        
        for i in range(len(class_names)):
            # Check if class has both positive (1) and negative (0) samples in this validation batch
            if len(np.unique(all_labels[:, i])) > 1:
                roc_score = roc_auc_score(all_labels[:, i], all_preds[:, i])
                ap_score = average_precision_score(all_labels[:, i], all_preds[:, i])
            else:
                roc_score = np.nan
                ap_score = np.nan
                
            per_class_roc_auc.append(roc_score)
            per_class_ap.append(ap_score)

        # Calculate Macro averages ignoring the NaNs
        valid_roc_scores = [s for s in per_class_roc_auc if not np.isnan(s)]
        valid_ap_scores = [s for s in per_class_ap if not np.isnan(s)]
        
        macro_roc_auc = np.mean(valid_roc_scores) if valid_roc_scores else np.nan
        macro_ap = np.mean(valid_ap_scores) if valid_ap_scores else np.nan

        # NEW: Delete the stacked numpy arrays once metrics are printed
        del all_labels, all_preds
        gc.collect()
        
        print(f"Validation Loss: {valid_loss:.4f} | Macro ROC-AUC: {macro_roc_auc:.4f} | Macro Avg Precision: {macro_ap:.4f}")
        
        # Print per-class breakdown
        print(f"{'Class':<28} | {'ROC-AUC':<8} | {'Avg Precision':<13}")
        print("-" * 55)
        for i, name in enumerate(class_names):
            roc_str = f"{per_class_roc_auc[i]:.4f}" if not np.isnan(per_class_roc_auc[i]) else "NaN"
            ap_str = f"{per_class_ap[i]:.4f}" if not np.isnan(per_class_ap[i]) else "NaN"
            print(f"{name:<28} | {roc_str:<8} | {ap_str:<13}")

        # =======================
        # SCHEDULER & CHECKPOINTING
        # =======================
        if scheduler is not None:
            scheduler.step()

        #Save standard PyTorch checkpoint
        save_path = f"/kaggle/working/efficientnet-b0-epoch-{epoch+1}.pth"
        print(f"\nSaving checkpoint to {save_path}...")
        torch.save(model.state_dict(), save_path)

In [11]:
train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    scheduler=None,  # Pass `None` if not using a scheduler
    criterion=criterion,
    device=device,
    epochs=5
)


--- Epoch 1/5 ---
Training Loss: 0.0881
Validation Loss: 0.0581 | Macro ROC-AUC: 0.5573 | Macro Avg Precision: 0.3045
Class                        | ROC-AUC  | Avg Precision
-------------------------------------------------------
No Finding                   | 0.6716   | 0.2803       
Enlarged Cardiomediastinum   | 0.5655   | 0.5319       
Cardiomegaly                 | 0.5827   | 0.3924       
Lung Opacity                 | 0.6605   | 0.7125       
Lung Lesion                  | 0.5494   | 0.0094       
Edema                        | 0.6167   | 0.3073       
Consolidation                | 0.4956   | 0.1498       
Pneumonia                    | 0.3252   | 0.0349       
Atelectasis                  | 0.4761   | 0.3447       
Pneumothorax                 | 0.6773   | 0.1543       
Pleural Effusion             | 0.6633   | 0.4523       
Pleural Other                | 0.3047   | 0.0061       
Fracture                     | NaN      | NaN          
Support Devices              | 0.6562   |